
# Fine-Tuning a DistilBERT Model for Custom Error Classification

**Author:** Dashgin Khudiyev
**Date:** June 14, 2025
**Description:** This notebook walks through the end-to-end process of fine-tuning a `distilbert-base-uncased` model on the `nkazi/SciEntsBank` dataset. It maps the original labels to a custom 5-way pedagogical schema to create a specialized error classifier.


## Step 1: Setup and Imports
First, we import all the necessary libraries. We'll need `datasets` to load our data, `transformers` for the model and training, `torch` for the underlying tensor operations, and `sklearn` for calculating detailed evaluation metrics.


In [1]:

import torch
import numpy as np
from datasets import load_dataset, ClassLabel
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    EvalPrediction
)
from sklearn.metrics import precision_recall_fscore_support, classification_report
import warnings

# Suppress irrelevant warnings from huggingface_hub
warnings.filterwarnings("ignore", category=FutureWarning)


## Step 2: Configuration and Constants
We define all our key parameters in one place. This includes the model we want to use, hyperparameters like batch size and learning rate, and the directory where the final model will be saved.


In [2]:

MODEL_CHECKPOINT = 'bert-base-multilingual-uncased'
BATCH_SIZE = 16
LEARNING_RATE = 3e-5
NUM_EPOCHS = 5
OUTPUT_DIR = './results/scientsbank-custom-classifier'
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {DEVICE}")


Using device: cpu


## Step 3: Load the Dataset
We load the `nkazi/SciEntsBank` dataset directly from the Hugging Face Hub. This dataset contains all the splits we need: `train`, `test_ua` (unseen answers), `test_uq` (unseen questions), and `test_ud` (unseen domains).


In [3]:
try:
    dataset = load_dataset('nkazi/SciEntsBank')
    print("Dataset loaded successfully:")
    print(dataset)
except Exception as e:
    print(f"Error loading dataset. Please check your internet connection. Error: {e}")


Using the latest cached version of the dataset since nkazi/SciEntsBank couldn't be found on the Hugging Face Hub
Found the latest cached dataset configuration 'default' at /Users/dashgin/.cache/huggingface/datasets/nkazi___sci_ents_bank/default/0.0.0/abaadf77345c5d68b73b630131a8ae164a45f3ab (last modified on Sun Jun 15 01:41:37 2025).


Dataset loaded successfully:
DatasetDict({
    train: Dataset({
        features: ['id', 'question', 'reference_answer', 'student_answer', 'label'],
        num_rows: 4969
    })
    test_ua: Dataset({
        features: ['id', 'question', 'reference_answer', 'student_answer', 'label'],
        num_rows: 540
    })
    test_uq: Dataset({
        features: ['id', 'question', 'reference_answer', 'student_answer', 'label'],
        num_rows: 733
    })
    test_ud: Dataset({
        features: ['id', 'question', 'reference_answer', 'student_answer', 'label'],
        num_rows: 4562
    })
})



## Step 4: Map Labels to Custom Schema
The original dataset has five labels. We will map these one-to-one to our custom pedagogical schema.
Note: The `logical_fallacy` category is not included in the training as there is no corresponding source label in the `SciEntsBank` dataset.

In [19]:
# Define the custom 5-way labels for our specific task.
custom_label_names = [
    'no_error',
    'factual_inaccuracy',
    'conceptual_misunderstanding',
    'incomplete_explanation',
    'irrelevant_content'
]
custom_labels = ClassLabel(names=custom_label_names)
num_labels = custom_labels.num_classes

# Get the original labels to create the one-to-one mapping
original_labels = dataset['train'].features['label'].names
label_map = {
    original_labels.index('correct'): custom_labels.str2int('no_error'),
    original_labels.index('contradictory'): custom_labels.str2int('factual_inaccuracy'),
    original_labels.index('partially_correct_incomplete'): custom_labels.str2int('incomplete_explanation'),
    original_labels.index('irrelevant'): custom_labels.str2int('irrelevant_content'),
    original_labels.index('non_domain'): custom_labels.str2int('conceptual_misunderstanding'),
}

def map_labels(example):
    """Applies the custom one-to-one label mapping to a dataset example."""
    original_label_id = example['label']
    example['label'] = label_map.get(original_label_id, -1) # Use -1 for any unexpected labels
    return example

# Apply the mapping across all splits and update the 'label' feature.
dataset = dataset.map(map_labels)
dataset = dataset.cast_column('label', custom_labels)

print("Label mapping complete. New label distribution:")
for split_name in dataset:
    print(f"  Split '{split_name}':")
    counts = {label: 0 for label in custom_labels.names}
    for label_id in dataset[split_name]['label']:
        if label_id != -1: # Ensure we don't count error labels
             counts[custom_labels.int2str(label_id)] += 1
    print(f"    {counts}")


Label mapping complete. New label distribution:
  Split 'train':
    {'no_error': 2008, 'factual_inaccuracy': 499, 'conceptual_misunderstanding': 23, 'incomplete_explanation': 1324, 'irrelevant_content': 1115}
  Split 'test_ua':
    {'no_error': 233, 'factual_inaccuracy': 58, 'conceptual_misunderstanding': 3, 'incomplete_explanation': 113, 'irrelevant_content': 133}
  Split 'test_uq':
    {'no_error': 301, 'factual_inaccuracy': 64, 'conceptual_misunderstanding': 0, 'incomplete_explanation': 175, 'irrelevant_content': 193}
  Split 'test_ud':
    {'no_error': 1917, 'factual_inaccuracy': 417, 'conceptual_misunderstanding': 20, 'incomplete_explanation': 986, 'irrelevant_content': 1222}


## Step 5: Preprocessing and Tokenization
We'll now load the tokenizer for our chosen model (`distilbert-base-uncased`) and use it to convert the `student_answer` text into numerical IDs that the model can understand.

In [20]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_CHECKPOINT)

def preprocess_function(examples):
    """Tokenizes student answers, applying lowercasing and padding."""
    return tokenizer(
        [text.lower() for text in examples['student_answer']],
        truncation=True,
        padding="max_length",
        max_length=128
    )

tokenized_dataset = dataset.map(preprocess_function, batched=True)
# We can remove the original text columns as they are no longer needed for training
tokenized_dataset = tokenized_dataset.remove_columns(['student_answer', 'id', 'question', 'reference_answer'])


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/625 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/872k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.72M [00:00<?, ?B/s]

Map:   0%|          | 0/4969 [00:00<?, ? examples/s]

Map:   0%|          | 0/540 [00:00<?, ? examples/s]

Map:   0%|          | 0/733 [00:00<?, ? examples/s]

Map:   0%|          | 0/4562 [00:00<?, ? examples/s]

## Step 6: Configure the Model
We load the `distilbert-base-uncased` model and configure it for sequence classification with the correct number of output labels (5 in our case). We also provide mappings between the label IDs and their string names.


In [22]:
# Create label mappings for the model configuration
id2label = {i: label for i, label in enumerate(custom_labels.names)}
label2id = {label: i for i, label in id2label.items()}

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_CHECKPOINT,
    num_labels=num_labels,
    id2label=id2label,
    label2id=label2id
).to(DEVICE)


model.safetensors:   0%|          | 0.00/672M [00:00<?, ?B/s]

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-multilingual-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


## Step 7: Define Evaluation Metrics
This function will be called by the `Trainer` at the end of each evaluation run. It calculates and returns the key performance metrics (Precision, Recall, and F1-Score) that we need to assess our model.

In [23]:
def compute_metrics(p: EvalPrediction):
    """Computes and returns evaluation metrics from model predictions."""
    preds = np.argmax(p.predictions, axis=1)
    labels = p.label_ids
    precision, recall, fscore, _ = precision_recall_fscore_support(
        labels, preds, average='macro', zero_division=0
    )
    return {
        'f1_macro': fscore,
        'precision_macro': precision,
        'recall_macro': recall,
    }

## Step 8: Configure and Run the Trainer
We set up the `TrainingArguments` with our hyperparameters and configure the `Trainer`. We also create a small validation set from our training data to monitor for overfitting during the training process. Finally, we call `trainer.train()` to start the fine-tuning.


In [24]:
# Reserve a small portion of the training set for validation
train_splits = tokenized_dataset["train"].train_test_split(test_size=0.1, seed=42)
train_dataset = train_splits['train']
eval_dataset = train_splits['test']

training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    learning_rate=LEARNING_RATE,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    num_train_epochs=NUM_EPOCHS,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1_macro",
    push_to_hub=False,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    processing_class=tokenizer,
    compute_metrics=compute_metrics,
)

print("Starting the training process...")
trainer.train()
print("Training complete.")

Starting the training process...


Epoch,Training Loss,Validation Loss,F1 Macro,Precision Macro,Recall Macro
1,No log,1.202557,0.244232,0.286806,0.284450
2,1.166800,1.148536,0.344717,0.449973,0.335898
3,1.166800,1.157128,0.503023,0.607172,0.459858


Training complete.


## Step 9: Final Evaluation
With the model trained, we now evaluate its performance on the three separate test sets. This gives us a clear picture of how well the model generalizes to unseen answers, unseen questions, and even unseen science domains.

In [25]:
print("--- Starting Final Evaluation on Test Sets ---")
tokenized_dataset
test_sets = {
    "Unseen Answers (test_ua)": tokenized_dataset['test_ua'],
    "Unseen Questions (test_uq)": tokenized_dataset['test_uq'],
    "Unseen Domains (test_ud)": tokenized_dataset['test_ud'],
}

for name, test_data in test_sets.items():
    print(f"\n--- Evaluating on: {name} ---")
    results = trainer.evaluate(test_data)
    print(f"  Macro F1-Score: {results['eval_f1_macro']:.4f}")
    print(f"  Precision: {results['eval_precision_macro']:.4f}")
    print(f"  Recall: {results['eval_recall_macro']:.4f}")

--- Starting Final Evaluation on Test Sets ---

--- Evaluating on: Unseen Answers (test_ua) ---


  Macro F1-Score: 0.4875
  Precision: 0.6319
  Recall: 0.4415

--- Evaluating on: Unseen Questions (test_uq) ---
  Macro F1-Score: 0.2587
  Precision: 0.2736
  Recall: 0.2642

--- Evaluating on: Unseen Domains (test_ud) ---
  Macro F1-Score: 0.3758
  Precision: 0.4673
  Recall: 0.3450


## Step 10: Generate Detailed Classification Report
To get a per-category breakdown of performance, we use `sklearn.metrics.classification_report`. This will produce a table showing Precision, Recall, and F1-score for each of our custom error classes. We'll generate this for the most important test set, `test_uq`.


In [26]:
# distilbert-base-uncased
"""

--- Detailed Classification Report for 'Unseen Questions (test_uq)' ---
                             precision    recall  f1-score   support

                   no_error       0.43      0.51      0.47       301
         factual_inaccuracy       0.22      0.12      0.16        64
conceptual_misunderstanding       0.00      0.00      0.00         0
     incomplete_explanation       0.33      0.18      0.23       175
         irrelevant_content       0.28      0.35      0.31       193

                   accuracy                           0.36       733
                  macro avg       0.25      0.23      0.23       733
               weighted avg       0.35      0.36      0.34       733

"""

"\n\n--- Detailed Classification Report for 'Unseen Questions (test_uq)' ---\n                             precision    recall  f1-score   support\n\n                   no_error       0.43      0.51      0.47       301\n         factual_inaccuracy       0.22      0.12      0.16        64\nconceptual_misunderstanding       0.00      0.00      0.00         0\n     incomplete_explanation       0.33      0.18      0.23       175\n         irrelevant_content       0.28      0.35      0.31       193\n\n                   accuracy                           0.36       733\n                  macro avg       0.25      0.23      0.23       733\n               weighted avg       0.35      0.36      0.34       733\n\n"

In [28]:
print("\n--- Detailed Classification Report for 'Unseen Questions (test_uq)' ---")
test_dataset_for_report = tokenized_dataset['test_ud']
predictions = trainer.predict(test_dataset_for_report)
predicted_labels = np.argmax(predictions.predictions, axis=1)
true_labels = predictions.label_ids

# Specify all possible labels explicitly to handle cases where some classes have 0 instances
all_labels = list(range(len(custom_labels.names)))

report = classification_report(
    true_labels,
    predicted_labels,
    labels=all_labels,
    target_names=custom_labels.names,
    zero_division=0
)

print(report)


--- Detailed Classification Report for 'Unseen Questions (test_uq)' ---


                             precision    recall  f1-score   support

                   no_error       0.50      0.55      0.52      1917
         factual_inaccuracy       0.33      0.11      0.17       417
conceptual_misunderstanding       0.89      0.40      0.55        20
     incomplete_explanation       0.21      0.19      0.20       986
         irrelevant_content       0.40      0.48      0.44      1222

                   accuracy                           0.41      4562
                  macro avg       0.47      0.34      0.38      4562
               weighted avg       0.40      0.41      0.40      4562



## Step 11: Save the Final Model
Finally, we save the best-performing model (and its tokenizer) to the specified output directory. This creates the final artifact that can be loaded into the AI Feedback Service for production inference.

In [ ]:
print(f"Saving the best model to '{OUTPUT_DIR}'...")
trainer.save_model(OUTPUT_DIR)
print("Model saved successfully.")
print("\n[SUCCESS] End-to-end training and evaluation pipeline complete.")